# Merge Fact Layer Chunks (v_n) — 4 Drive folders → 1 output folder

Pulls the per-case JSONs from the 4 team members' Drive folders (by link), validates them,
de-duplicates, and writes one merged folder ready for `clean_fact_layer.py`.

**Before running:** each person must share their output folder with the Google account you use in Colab (Viewer is enough).

## Cell 1 — Mount Drive + authenticate for the Drive API

In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()          # lets us read folders shared with you by link

from googleapiclient.discovery import build
print("Authenticated.")

## Cell 2 — Config (paste the 4 folder links here)

In [ ]:
import re

CHUNK_LINKS = {
    "chunk1": "https://drive.google.com/drive/folders/PASTE_LINK_1",
    "chunk2": "https://drive.google.com/drive/folders/PASTE_LINK_2",
    "chunk3": "https://drive.google.com/drive/folders/PASTE_LINK_3",
    "chunk4": "https://drive.google.com/drive/folders/PASTE_LINK_4",
}

OUT_DIR = "/content/drive/MyDrive/MERGED_fact_layer_final_v2"   # merged output
N_WORKERS = 8                                                   # parallel downloads
EXPECTED_TOTAL = None   # optional: set to the number of year>=2000 cases to get a missing-count check

# Files that live in the output folders but are not case JSONs
SKIP_NAMES = {"done_keys.json", "errors.json", "checkpoint.json", "token_usage.json"}

def folder_id(link):
    m = re.search(r"/folders/([A-Za-z0-9_-]+)", link) or re.search(r"[?&]id=([A-Za-z0-9_-]+)", link)
    if m: return m.group(1)
    if re.fullmatch(r"[A-Za-z0-9_-]{20,}", link.strip()): return link.strip()   # bare ID
    raise ValueError(f"Can't parse folder id from: {link}")

CHUNK_IDS = {k: folder_id(v) for k, v in CHUNK_LINKS.items()}
print(CHUNK_IDS)

## Cell 3 — List every JSON in each folder (recursive)

In [ ]:
def list_json_files(service, root_id):
    """Recursively list .json files under a Drive folder."""
    out, stack = [], [root_id]
    while stack:
        fid = stack.pop()
        token = None
        while True:
            resp = service.files().list(
                q=f"'{fid}' in parents and trashed=false",
                fields="nextPageToken, files(id,name,mimeType,size)",
                pageSize=1000, pageToken=token,
                supportsAllDrives=True, includeItemsFromAllDrives=True,
            ).execute()
            for f in resp.get("files", []):
                if f["mimeType"] == "application/vnd.google-apps.folder":
                    stack.append(f["id"])
                elif f["name"].lower().endswith(".json") and f["name"] not in SKIP_NAMES:
                    out.append(f)
            token = resp.get("nextPageToken")
            if not token: break
    return out

svc = build("drive", "v3", cache_discovery=False)
LISTING = {}
for name, fid in CHUNK_IDS.items():
    LISTING[name] = list_json_files(svc, fid)
    print(f"{name}: {len(LISTING[name]):>6} JSON files")
print("Total listed:", sum(len(v) for v in LISTING.values()))

## Cell 4 — Download, validate, de-duplicate, write

- **Validation:** must parse as JSON, be a dict, and contain `case_id`. Anything else is logged as `invalid`.
- **Duplicates** (same `case_id` in more than one chunk): keep the record whose `extraction_method` has no `regex_fallback`; otherwise first seen wins.
- **Resumable:** files already in `OUT_DIR` are skipped, so just re-run if Colab disconnects.

In [ ]:
import os, json, time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter, defaultdict
from tqdm.auto import tqdm

os.makedirs(OUT_DIR, exist_ok=True)
already = set(os.listdir(OUT_DIR))

_tls = threading.local()
def get_service():
    if not hasattr(_tls, "svc"):
        _tls.svc = build("drive", "v3", cache_discovery=False)
    return _tls.svc

def download_bytes(file_id, retries=5):
    for i in range(retries):
        try:
            return get_service().files().get_media(fileId=file_id, supportsAllDrives=True).execute()
        except Exception as e:
            if i == retries - 1: raise
            time.sleep(2 ** i)

def is_fallback(rec):
    return "regex_fallback" in str(rec.get("extraction_method", ""))

lock = threading.Lock()
seen = {}                      # case_id -> {"name":..., "chunk":..., "fallback":bool}
stats = Counter()
per_chunk = defaultdict(Counter)
dup_log, invalid_log, fail_log = [], [], []

def handle(chunk, f):
    name = f["name"]
    if name in already:
        return chunk, "skipped_existing", None
    try:
        raw = download_bytes(f["id"])
        rec = json.loads(raw)
    except json.JSONDecodeError:
        return chunk, "invalid", (chunk, name, "bad JSON")
    except Exception as e:
        return chunk, "failed", (chunk, name, repr(e))
    if not isinstance(rec, dict) or "case_id" not in rec:
        return chunk, "invalid", (chunk, name, "no case_id / not a case record")

    cid, fb = rec["case_id"], is_fallback(rec)
    with lock:
        prev = seen.get(cid)
        if prev is None:
            seen[cid] = {"name": name, "chunk": chunk, "fallback": fb}
            write = True; status = "written"
        elif prev["fallback"] and not fb:           # new one is better -> replace
            if prev["name"] != name:
                try: os.remove(os.path.join(OUT_DIR, prev["name"]))
                except FileNotFoundError: pass
            dup_log.append((cid, prev["chunk"], chunk, "replaced_fallback"))
            seen[cid] = {"name": name, "chunk": chunk, "fallback": fb}
            write = True; status = "replaced"
        else:
            dup_log.append((cid, prev["chunk"], chunk, "kept_first"))
            write = False; status = "duplicate"
    if write:
        with open(os.path.join(OUT_DIR, name), "wb") as fh:
            fh.write(raw)
    return chunk, status, (("fallback", chunk, name) if (write and fb) else None)

jobs = [(c, f) for c, files in LISTING.items() for f in files]
fallback_files = []
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futs = [ex.submit(handle, c, f) for c, f in jobs]
    for fu in tqdm(as_completed(futs), total=len(futs)):
        chunk, status, extra = fu.result()
        stats[status] += 1
        per_chunk[chunk][status] += 1
        if status == "invalid": invalid_log.append(extra)
        elif status == "failed": fail_log.append(extra)
        elif extra and extra[0] == "fallback": fallback_files.append(extra[1:])

print(dict(stats))

## Cell 5 — Merge report + manifest

In [ ]:
final_files = [f for f in os.listdir(OUT_DIR) if f.endswith(".json") and f not in SKIP_NAMES and f != "merge_manifest.json"]

print("=== PER-CHUNK ===")
for c in CHUNK_LINKS:
    print(f"  {c}: listed={len(LISTING[c])}  {dict(per_chunk[c])}")

print("\n=== TOTALS ===")
print(f"  Files in OUT_DIR       : {len(final_files)}")
print(f"  Duplicate case_ids     : {len(dup_log)}")
print(f"  Invalid / non-case     : {len(invalid_log)}")
print(f"  Download failures      : {len(fail_log)}   <-- re-run Cell 4 if > 0")
print(f"  regex_fallback records : {len(fallback_files)}  (partial/failed LLM extraction)")
if EXPECTED_TOTAL:
    print(f"  Missing vs expected    : {EXPECTED_TOTAL - len(final_files)}")

yr = Counter(f.split("_")[0] for f in final_files)
print("\n=== YEAR RANGE ===")
print(" ", min(yr), "→", max(yr), "|", sum(v for k, v in yr.items() if k.isdigit() and int(k) < 2000), "cases before 2000 (should be 0)")

manifest = {
    "total_files": len(final_files),
    "per_chunk_listed": {c: len(LISTING[c]) for c in LISTING},
    "duplicates": dup_log, "invalid": invalid_log, "failures": fail_log,
    "fallback_files": fallback_files,
}
with open(os.path.join(OUT_DIR, "merge_manifest.json"), "w") as fh:
    json.dump(manifest, fh, indent=2)
print("\nManifest saved →", os.path.join(OUT_DIR, "merge_manifest.json"))

## Cell 6 — Spot check a random merged case

In [ ]:
import random
f = random.choice(final_files)
with open(os.path.join(OUT_DIR, f)) as fh:
    r = json.load(fh)
print(f)
print("case_id          :", r.get("case_id"))
print("extraction_method:", r.get("extraction_method"))
print("outcome          :", (r.get("decision") or {}).get("outcome") or r.get("outcome"))
print("keys             :", list(r.keys())[:12])

### Next steps (from the handoff)
1. Point `clean_fact_layer.py` at `OUT_DIR` → writes `CLEANED_fact_layer/`
2. Rebuild the KG (`KG_Construction.ipynb` Cell 5, which clears the graph first)
3. Recompute `SIMILAR_FACTS` (Cell 6)
4. Re-run the `regex_fallback` cases listed in `merge_manifest.json` if the count is meaningful